# Notebook 05: Classification and Clustering

Demonstrates KNN, SVM, and Ward hierarchical clustering on adsorption descriptor data using scikit-learn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn.metrics import confusion_matrix, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

## Generate synthetic descriptor dataset

Three groups of materials in {Q_max, ln K*_aff, sigma_E} space.

In [ ]:
rng = np.random.default_rng(42)

# Group A: low capacity, high affinity
A = np.column_stack([
    rng.normal(50, 15, 10),
    rng.normal(3.0, 0.5, 10),
    rng.normal(2.0, 0.5, 10)
])

# Group B: high capacity, low affinity
B = np.column_stack([
    rng.normal(200, 30, 10),
    rng.normal(0.5, 0.3, 10),
    rng.normal(5.0, 1.0, 10)
])

# Group C: high capacity, high affinity
C = np.column_stack([
    rng.normal(180, 25, 10),
    rng.normal(2.5, 0.4, 10),
    rng.normal(1.0, 0.3, 10)
])

X = np.vstack([A, B, C])
y = np.array([0]*10 + [1]*10 + [2]*10)
group_names = ['Group A', 'Group B', 'Group C']

df = pd.DataFrame(X, columns=['Qmax', 'ln_Kaff', 'sigma_E'])
df['Group'] = [group_names[i] for i in y]
df.head(10)

## KNN classification with leave-one-out cross-validation

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

knn = KNeighborsClassifier(n_neighbors=5)
loo = LeaveOneOut()
scores = cross_val_score(knn, X_scaled, y, cv=loo)
print(f'KNN (k=5) LOO accuracy: {scores.mean():.2%}')

# Fit on full data for plotting
knn.fit(X_scaled[:, :2], y)  # 2D projection

# Decision region
fig, ax = plt.subplots(figsize=(8, 6))
x_min, x_max = X_scaled[:, 0].min() - 1, X_scaled[:, 0].max() + 1
y_min, y_max = X_scaled[:, 1].min() - 1, X_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                      np.linspace(y_min, y_max, 200))
Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

ax.contourf(xx, yy, Z, alpha=0.2, cmap='Set2')
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for i, name in enumerate(group_names):
    mask = y == i
    ax.scatter(X_scaled[mask, 0], X_scaled[mask, 1],
               c=colors[i], label=name, edgecolors='k', s=60)

ax.set_xlabel('Q_max (standardised)')
ax.set_ylabel('ln K*_aff (standardised)')
ax.set_title('KNN Decision Regions (k=5)')
ax.legend()
plt.tight_layout()
plt.show()

## SVM classification

In [ ]:
svm = SVC(kernel='linear', C=1.0)
svm_scores = cross_val_score(svm, X_scaled[:, :2], y, cv=5)
print(f'SVM (linear) 5-fold CV accuracy: {svm_scores.mean():.2%}')

svm.fit(X_scaled[:, :2], y)

fig, ax = plt.subplots(figsize=(8, 6))
Z_svm = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z_svm, alpha=0.2, cmap='Set2')
ax.contour(xx, yy, Z_svm, colors='k', linewidths=0.5)

for i, name in enumerate(group_names):
    mask = y == i
    ax.scatter(X_scaled[mask, 0], X_scaled[mask, 1],
               c=colors[i], label=name, edgecolors='k', s=60)

# Mark support vectors
ax.scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1],
           s=150, facecolors='none', edgecolors='red', linewidths=2,
           label='Support vectors')

ax.set_xlabel('Q_max (standardised)')
ax.set_ylabel('ln K*_aff (standardised)')
ax.set_title('SVM Decision Boundary (linear kernel)')
ax.legend()
plt.tight_layout()
plt.show()

## Ward hierarchical clustering

In [ ]:
Z_link = linkage(X_scaled, method='ward')

fig, ax = plt.subplots(figsize=(10, 5))
dendrogram(Z_link, ax=ax, color_threshold=5.0,
           labels=[f'{group_names[yi]}-{i}' for i, yi in enumerate(y)])
ax.axhline(y=5.0, color='r', linestyle='--', label='Cut threshold')
ax.set_ylabel('Ward linkage distance')
ax.set_title('Hierarchical Clustering Dendrogram')
ax.legend()
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

## Cluster validation

In [ ]:
cluster_labels = fcluster(Z_link, t=3, criterion='maxclust')

print('Confusion matrix (true groups vs clusters):')
print(confusion_matrix(y, cluster_labels - 1))
print(f'\nAdjusted Rand Index: {adjusted_rand_score(y, cluster_labels):.3f}')

# 3D scatter
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

for i, name in enumerate(group_names):
    mask = y == i
    ax.scatter(X[mask, 0], X[mask, 1], X[mask, 2],
               c=colors[i], label=name, s=60, edgecolors='k')

ax.set_xlabel('Q_max (mg/g)')
ax.set_ylabel('ln K*_aff')
ax.set_zlabel('sigma_E (kJ/mol)')
ax.set_title('3D Descriptor Space')
ax.legend()
plt.tight_layout()
plt.show()